# Hybrid Quantum Vision Transformer vs. Classical CNN

**Goal:** Achieve >90% accuracy on Arabic Handwritten Character Recognition (AHCD).

**Architecture Comparison:**
1. **Classical CNN:** Standard Convolutional Neural Network baseline.
2. **Hybrid CNN-QViT:** High-performance model using Classical Convolutions for fast spatial feature extraction, followed by a **Quantum Self-Attention** layer for global context.

Run all cells sequentially. We use **16x16 resolution** and the **full 13,440 dataset**.

In [8]:
# Cell 1: Install dependencies
!pip install -q pennylane pennylane-lightning torch torchvision scikit-learn matplotlib seaborn kagglehub tqdm

In [9]:
# Cell 2: Clone repo
!git clone https://github.com/aminemons/Quantum-Vision-Transformer-Arabic-OCR.git qcnn_vit 2>/dev/null || echo 'Already cloned'
import os
if os.path.basename(os.getcwd()) != 'qcnn_vit': os.chdir('qcnn_vit')
!git pull
!ls src/

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 927 bytes | 231.00 KiB/s, done.
From https://github.com/aminemons/Quantum-Vision-Transformer-Arabic-OCR
   8a3b5ca..ac8297d  master     -> origin/master
Updating 8a3b5ca..ac8297d
Fast-forward
 src/quantum_attention.py | 45 ++++++++++++++++++++++++++++++---------------
 1 file changed, 30 insertions(+), 15 deletions(-)
classical_cnn.py  __init__.py	 quantum_attention.py
data_loader.py	  __pycache__	 train.py
hybrid_qvit.py	  qcnn_layer.py  visualize.py


In [10]:
# Cell 3: Imports & Setup
import sys, os, json, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

sys.path.insert(0, 'src')
print(f'PyTorch {torch.__version__} | PennyLane {qml.__version__}')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Device: {device}')

PyTorch 2.10.0+cpu | PennyLane 0.44.1
Device: cpu


In [11]:
# Cell 4: Download & Load AHCD Dataset (16x16 Resolution)
import kagglehub, pandas as pd
from torch.utils.data import Dataset, DataLoader, Subset

ARABIC_CHARS = ['Alef','Beh','Teh','Theh','Jeem','Hah','Khah','Dal','Thal','Reh',
                'Zain','Seen','Sheen','Sad','Dad','Tah','Zah','Ain','Ghain','Feh',
                'Qaf','Kaf','Lam','Meem','Noon','Heh','Waw','Yeh']
NUM_CLASSES = 28; IMG_SIZE = 16

# Download
path = kagglehub.dataset_download('mloey1/ahcd1')
csv_dir = [root for root, _, files in os.walk(path) if any('trainimages' in f.lower().replace(' ','') for f in files)][0]

def find_csv(d, key):
    return [os.path.join(d, f) for f in os.listdir(d) if key in f.lower().replace(' ','') and f.endswith('.csv')][0]

t0 = time.time()
train_img = pd.read_csv(find_csv(csv_dir,'trainimages'), header=None).values.astype(np.float32)
train_lbl = pd.read_csv(find_csv(csv_dir,'trainlabel'), header=None).values.flatten().astype(int) - 1
test_img = pd.read_csv(find_csv(csv_dir,'testimages'), header=None).values.astype(np.float32)
test_lbl = pd.read_csv(find_csv(csv_dir,'testlabel'), header=None).values.flatten().astype(int) - 1

print(f'Loaded in {time.time()-t0:.1f}s: {train_img.shape[0]} train, {test_img.shape[0]} test')

from skimage.transform import resize as sk_resize
def fast_resize(images, orig=32, target=16):
    n = len(images)
    imgs = images.reshape(n, orig, orig)
    resized = np.zeros((n, target, target), dtype=np.float32)
    for i in range(n):
        resized[i] = sk_resize(imgs[i], (target, target), anti_aliasing=True, preserve_range=True)
    return resized.reshape(n, -1)

def normalize(images):
    lo = images.min(axis=1, keepdims=True)
    hi = images.max(axis=1, keepdims=True)
    denom = np.where(hi - lo > 1e-8, hi - lo, 1.0)
    return (images - lo) / denom  # Scale 0 to 1

print('Resizing images to 16x16...', flush=True)
train_img_r = normalize(fast_resize(train_img, target=IMG_SIZE))
test_img_r = normalize(fast_resize(test_img, target=IMG_SIZE))

class ArabicDS(Dataset):
    def __init__(self, imgs, lbls):
        self.X = torch.tensor(imgs, dtype=torch.float32)
        self.y = torch.tensor(lbls, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

# Split 85% Train / 15% Val
tr_idx, val_idx = train_test_split(np.arange(len(train_img_r)), test_size=0.15, stratify=train_lbl, random_state=42)
full_ds = ArabicDS(train_img_r, train_lbl)
train_loader = DataLoader(Subset(full_ds, tr_idx), batch_size=64, shuffle=True)
val_loader = DataLoader(Subset(full_ds, val_idx), batch_size=64)
test_loader = DataLoader(ArabicDS(test_img_r, test_lbl), batch_size=128)
print(f'Loaders: {len(train_loader)} train / {len(val_loader)} val / {len(test_loader)} test batches')

Using Colab cache for faster access to the 'ahcd1' dataset.
Loaded in 1.8s: 13440 train, 3360 test
Resizing images to 16x16...
Loaders: 179 train / 32 val / 27 test batches


In [13]:
# Cell 5: Define Training Logic
def train_model(model, name, epochs=10, lr=0.002):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
    best_val_acc = 0
    os.makedirs('results', exist_ok=True)

    print(f"\n{'='*50}\n Training: {name}\n{'='*50}")

    for epoch in range(epochs):
        model.train()
        tot_loss = tot_acc = tot_n = 0
        t0 = time.time()

        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{epochs} [Train]", leave=False)
        for imgs, lbls in pbar:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            acc = (logits.argmax(1) == lbls).sum().item()
            tot_loss += loss.item()*len(imgs); tot_acc += acc; tot_n += len(imgs)
            pbar.set_postfix({'loss': f"{loss.item():.3f}"})

        tr_loss, tr_acc = tot_loss/tot_n, tot_acc/tot_n

        model.eval()
        tot_loss = tot_acc = tot_n = 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                logits = model(imgs)
                loss = criterion(logits, lbls)
                tot_loss += loss.item()*len(imgs); tot_acc += (logits.argmax(1)==lbls).sum().item(); tot_n += len(imgs)
        vl_loss, vl_acc = tot_loss/tot_n, tot_acc/tot_n
        scheduler.step()

        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss); history['val_acc'].append(vl_acc)

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), f'results/best_{name}.pt')

        print(f"Epoch {epoch+1:2d} | Train Acc: {tr_acc:.4f} | Val Acc: {vl_acc:.4f} | Time: {time.time()-t0:.1f}s")

    return history

In [14]:
# Cell 6: Train Classical CNN Baseline
import importlib
import classical_cnn; importlib.reload(classical_cnn)
from classical_cnn import create_cnn_baseline

model_cnn = create_cnn_baseline(n_classes=28, img_size=16, device=device)
print(f"Classical CNN Params: {sum(p.numel() for p in model_cnn.parameters())}")

# Train CNN
history_cnn = train_model(model_cnn, "Classical_CNN", epochs=15, lr=0.003)

Classical CNN Params: 81020

 Training: Classical_CNN


Ep 1/15 [Train]:   0%|          | 0/179 [00:00<?, ?it/s]

Epoch  1 | Train Acc: 0.4638 | Val Acc: 0.7034 | Time: 4.5s


Ep 2/15 [Train]:   0%|          | 0/179 [00:00<?, ?it/s]

Epoch  2 | Train Acc: 0.7004 | Val Acc: 0.7842 | Time: 3.5s


...

Epoch 13 | Train Acc: 0.9342 | Val Acc: 0.9117 | Time: 3.5s


Ep 14/15 [Train]:   0%|          | 0/179 [00:00<?, ?it/s]

Epoch 14 | Train Acc: 0.9370 | Val Acc: 0.9107 | Time: 3.5s


Ep 15/15 [Train]:   0%|          | 0/179 [00:00<?, ?it/s]

Epoch 15 | Train Acc: 0.9436 | Val Acc: 0.9102 | Time: 4.4s


In [ ]:
# Cell 7: Train Hybrid CNN-QViT
import hybrid_qvit; importlib.reload(hybrid_qvit)
from hybrid_qvit import create_model

model_qvit = create_model(n_classes=28, img_size=16, device=device)
print(f"Hybrid CNN-QViT Params: {sum(p.numel() for p in model_qvit.parameters())}")

# Train Hybrid Model (uses Quantum circuits so it takes slightly longer, but much faster than v1)
history_qvit = train_model(model_qvit, "Hybrid_QViT", epochs=15, lr=0.003)

Hybrid CNN-QViT Params: 36798

 Training: Hybrid_QViT


Ep 1/15 [Train]:   0%|          | 0/179 [00:00<?, ?it/s]

In [ ]:
# Cell 8: Test Set Evaluation
def evaluate_test(model, name):
    model.load_state_dict(torch.load(f'results/best_{name}.pt', weights_only=True))
    model.eval()
    tot_acc = tot_n = 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            tot_acc += (logits.argmax(1) == lbls).sum().item()
            tot_n += len(imgs)
    acc = tot_acc / tot_n
    print(f"[{name}] Test Accuracy: {acc:.4f} ({(acc*100):.1f}%)")
    return acc

acc_cnn = evaluate_test(model_cnn, "Classical_CNN")
acc_qvit = evaluate_test(model_qvit, "Hybrid_QViT")

In [ ]:
# Cell 9: Plot Comparison
C = {'bg':'#0a0a1a','fg':'#e0e0ff','cnn':'#00d4ff','qvit':'#ff6b9d','grid':'#1a1a3e'}
plt.rcParams.update({'figure.facecolor':C['bg'],'axes.facecolor':C['bg'],'text.color':C['fg'],
    'axes.labelcolor':C['fg'],'xtick.color':C['fg'],'ytick.color':C['fg'],'axes.edgecolor':C['grid']})

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
epochs_range = range(1, len(history_cnn['val_acc'])+1)

ax.plot(epochs_range, history_cnn['val_acc'], '-o', color=C['cnn'], lw=2, label=f'Classical CNN (Test: {acc_cnn:.1%})')
ax.plot(epochs_range, history_qvit['val_acc'], '-s', color=C['qvit'], lw=2, label=f'Hybrid QViT (Test: {acc_qvit:.1%})')
ax.axhline(0.90, color='#34d399', ls='--', alpha=0.7, label='90% Target')

ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title('Classical vs. Hybrid Quantum Accuracy Comparison', fontsize=14, pad=15)
ax.legend(facecolor=C['bg'], edgecolor=C['grid'], fontsize=11)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()